# Track B: GNews MCP behind the APIM gateway

Put the GNews MCP server behind `apim-foundry-{suffix}`, the gateway 05-02 deployed, and enforce the same per-user rules in a gateway policy instead of a server you own. `news-desk-agent` keeps one MCP tool; only its connection changes, to `news-mcp-apim` from 08-05c-01.

The API is a pass-through. Everyone sees GNews's own two tools, `top_headlines(category, ...)` and `search(q, ...)`: a policy at MCP scope must not read the response body, so it cannot filter `tools/list`. The policy, [`infra/news-mcp-policy.xml`](infra/news-mcp-policy.xml), validates the Entra token Foundry sends on the user's behalf, reads the JSON-RPC body of each `tools/call`, and answers `top_headlines` for a category the token's roles do not allow, or `search` without `News.Read.All`, with the same denial text as the Track A wrapper. Allowed calls go to GNews with the key from a secret named value, and the user's token stops at the gateway. [08-05c-00](08-05c-00-news-mcp-per-user-access.md) compares the two tracks.

**What this notebook does:**
1. Adds the `news-mcp` MCP API, its policy, the `gnews-mcp` backend and the `gnews-api-key` named value to `apim-foundry-{suffix}`
2. Checks that a request without a token gets 401
3. Creates a `news-desk-agent` version whose tool uses the `news-mcp-apim` connection
4. Signs in user A and user B
5. Runs the test matrix as both users, after 08-05c-02 revoked user B's `News.Read.Science`
6. Sends user B's denied call straight to the gateway
7. Asks the Track A version for science as user B, to see whether the revocation has reached it

## Prerequisites

1. **08-05c-01 and 08-05c-02** have run. Step 10 of 08-05c-02 revoked user B's `News.Read.Science` and wrote the time to `.env` as `NEWS_SCIENCE_REVOKED_AT`. Run this notebook within the hour after that, so Step 6 can show whether Track A still honours a token Foundry obtained before the revocation.
2. **Core gateway**: [05-02](../../05-foundry-project-pattern-setup/05-02-deploy-foundry-core-gateway/05-02-01-deploy-foundry-core-gateway.ipynb) deployed `apim-foundry-{suffix}` in `rg-foundry-core-{suffix}`. This notebook adds one API, its policy, one backend and one named value, and changes nothing else on the gateway.
3. **`GNEWS_API_KEY`** in `.env`, as for 08-05c-02. This notebook makes two or three GNews requests; denied calls make none.
4. **Azure CLI** as in 08-05c-01, with rights to deploy into `rg-foundry-core-{suffix}`.
5. **User B's pass** from 08-05c-01 is valid for 8 hours. If it has expired, re-run Step 5 of 08-05c-01 for a new one.

## Step 0: Configuration

In [1]:
import hashlib, json, os, subprocess, tempfile, time
from datetime import datetime, timezone
from pathlib import Path

import requests
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

import news_mcp_checks as checks
import news_mcp_identity as ident
from news_mcp_identity import authz

repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
ENV_FILE = repo_root / '.env'
load_dotenv(ENV_FILE, override=True)

REQUIRED = ['NEWS_MCP_RESOURCE_APP_ID', 'NEWS_MCP_RESOURCE_SP_ID', 'NEWS_DESK_CLIENT_APP_ID', 'NEWS_TEST_USER_UPN',
            'NEWS_TEST_USER_ID', 'NEWS_AGENT_BLUEPRINT_CLIENT_ID', 'NEWS_CONNECTION_API_VERSION',
            'NEWS_MCP_WRAPPER_URL', 'NEWS_SCIENCE_REVOKED_AT', 'GNEWS_API_KEY']
missing = [key for key in REQUIRED if not os.environ.get(key)]
if missing:
    raise SystemExit(f"Missing in .env: {', '.join(missing)}. Run 08-05c-01 and 08-05c-02 first, and add GNEWS_API_KEY.")

account = json.loads(subprocess.run('az account show -o json', shell=True, capture_output=True, text=True).stdout)
SUBSCRIPTION_ID, TENANT_ID = account['id'], account['tenantId']
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-5.4-mini')
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

RESOURCE_APP_ID = os.environ['NEWS_MCP_RESOURCE_APP_ID']
RESOURCE_SP_ID = os.environ['NEWS_MCP_RESOURCE_SP_ID']
CLIENT_APP_ID = os.environ['NEWS_DESK_CLIENT_APP_ID']
BLUEPRINT_CLIENT_ID = os.environ['NEWS_AGENT_BLUEPRINT_CLIENT_ID']
TEST_USER_ID = os.environ['NEWS_TEST_USER_ID']
CONNECTION_API = os.environ['NEWS_CONNECTION_API_VERSION']
REVOKED_AT = datetime.fromisoformat(os.environ['NEWS_SCIENCE_REVOKED_AT'])

CORE_RG, APIM_NAME = f'rg-foundry-core-{SUFFIX}', f'apim-foundry-{SUFFIX}'
APIM_API_VERSION = '2024-06-01-preview'
APIM = f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
TRACK_A_RG = os.environ.get('NEWS_MCP_RESOURCE_GROUP') or f'rg-foundry-news-mcp-{SUFFIX}'
TRACK_A_APP = f'ca-news-mcp-{SUFFIX}'
LAB_DIR = Path.cwd()

cli = AzureCliCredential()
ARM, FOUNDRY = 'https://management.azure.com/.default', 'https://ai.azure.com/.default'
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cli, allow_preview=True)


def graph(method, path, body=None):
    """Microsoft Graph v1.0 as the Azure CLI user."""
    r = requests.request(method, f'https://graph.microsoft.com/v1.0{path}', json=body, timeout=60,
                         headers={'Authorization': f"Bearer {cli.get_token('https://graph.microsoft.com/.default').token}"})
    if not r.ok:
        raise RuntimeError(f'{method} {path}: HTTP {r.status_code} {r.text[:600]}')
    return r.json() if r.content else None


def call(method, url, scope, body=None):
    """ARM or the Foundry data plane with a CLI token for `scope`. The caller checks the response."""
    return requests.request(method, url, data=None if body is None else json.dumps(body), timeout=120,
                            headers={'Authorization': f'Bearer {cli.get_token(scope).token}', 'Content-Type': 'application/json'})


def az(*args):
    """Run an az command; return its JSON output, or raise with the end of stderr."""
    r = subprocess.run(['az', *args, '-o', 'json'], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"az {' '.join(args[:3])}: {r.stderr.strip()[-1500:]}")
    return json.loads(r.stdout) if r.stdout.strip() else None


def roles_of(principal_id):
    """The app roles a principal holds on news-mcp-{suffix} now, read from Graph."""
    held = {a['appRoleId'] for a in graph('GET', f'/servicePrincipals/{RESOURCE_SP_ID}/appRoleAssignedTo')['value']
            if a['principalId'] == principal_id}
    return [role for role in authz.ROLES if authz.permission_id(SUFFIX, role) in held]


def minutes_since_revocation():
    return (datetime.now(timezone.utc) - REVOKED_AT).total_seconds() / 60


print(f'Suffix          : {SUFFIX}')
print(f'Gateway         : {APIM_NAME} in {CORE_RG}')
print(f'Science revoked : {REVOKED_AT.isoformat()}, {minutes_since_revocation():.0f} minutes ago')

Suffix          : c2676f
Gateway         : apim-foundry-c2676f in rg-foundry-core-c2676f
Science revoked : 2026-10-30T21:01:51+00:00, 0 minutes ago


## Step 1: The MCP API on the gateway

[`infra/apim-news-mcp.bicep`](infra/apim-news-mcp.bicep) references `apim-foundry-{suffix}` as an existing resource and adds four things to it:

| Resource | Setting |
|---|---|
| named value `gnews-api-key` | secret; the policy sends it to GNews as `X-Api-Key` |
| backend `gnews-mcp` | `https://mcp.gnews.io/`, where GNews serves MCP |
| API `news-mcp` | `type: mcp`, path `news-mcp`, `backendId` `gnews-mcp`, streamable transport, no subscription key |
| API policy | the policy file, with the tenant and the two accepted audiences filled in |

The `backendId` is what makes the API a pass-through. Without it APIM answers `initialize` itself and lists the tools of the API's own operations, of which there are none. Clients reach the server at `/mcp` under the API's path, and APIM appends that `/mcp` to the backend URL; GNews has nothing there, so the policy ends with `rewrite-uri` to `/`.

The cell lists the gateway's APIs before and after, to show that the deployment added one and left the others as they were.

In [5]:
def apis():
    r = call('GET', f'{APIM}/apis?api-version={APIM_API_VERSION}', ARM)
    r.raise_for_status()
    return {a['name']: (a['properties'].get('type') or 'http', a['properties']['path']) for a in r.json()['value']}


before = apis()
params = {'suffix': SUFFIX, 'tenantId': TENANT_ID, 'audiences': [RESOURCE_APP_ID, authz.app_id_uri(SUFFIX)],
          'gnewsApiKey': os.environ['GNEWS_API_KEY']}
with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as f:
    json.dump({'$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
               'contentVersion': '1.0.0.0', 'parameters': {k: {'value': v} for k, v in params.items()}}, f)
t0 = time.time()
try:
    result = az('deployment', 'group', 'create', '-g', CORE_RG, '-n', 'news-mcp-apim',
                '--template-file', str(LAB_DIR / 'infra' / 'apim-news-mcp.bicep'), '--parameters', f'@{f.name}')
finally:
    os.unlink(f.name)
APIM_MCP_URL = result['properties']['outputs']['mcpUrl']['value']
print(f'Deployed in {time.time() - t0:.0f}s: {APIM_MCP_URL}\n')

api = call('GET', f'{APIM}/apis/news-mcp?api-version={APIM_API_VERSION}', ARM).json()['properties']
backend = call('GET', f"{APIM}/backends/{api['backendId']}?api-version={APIM_API_VERSION}", ARM).json()['properties']
print(f"news-mcp: type={api['type']}  path={api['path']}  backend={api['backendId']} ({backend['url']})  subscriptionRequired={api['subscriptionRequired']}")

after = apis()
print(f'\nAPIs on the gateway: {len(before)} before, {len(after)} after')
for name, (kind, path) in sorted(after.items()):
    state = 'added' if name not in before else 'unchanged' if before[name] == (kind, path) else 'CHANGED'
    print(f'  {name:<22} {kind:<5} /{path:<22} {state}')

Deployed in 36s: https://apim-foundry-c2676f.azure-api.net/news-mcp/mcp

news-mcp: type=mcp  path=news-mcp  backend=gnews-mcp (https://mcp.gnews.io/)  subscriptionRequired=False

APIs on the gateway: 1 before, 2 after
  news-mcp               mcp   /news-mcp               added
  openai                 http  /openai                 unchanged


## Step 2: No token, no entry

`validate-azure-ad-token` is the first thing the policy does, so a request without a bearer token stops at the gateway with 401. A new API can take a few seconds to answer after the deployment; the cell waits for up to two minutes.

In [6]:
for attempt in range(13):
    r = requests.post(APIM_MCP_URL, json={'jsonrpc': '2.0', 'id': 1, 'method': 'tools/list'},
                      headers={'Accept': 'application/json, text/event-stream'}, timeout=30)
    if r.status_code == 401:
        break
    time.sleep(10)
print(f'tools/list without a token: HTTP {r.status_code}')
assert r.status_code == 401

tools/list without a token: HTTP 401


## Step 3: A `news-desk-agent` version on the gateway

08-05c-01 created `news-mcp-apim` with the same delegated authentication as `news-mcp-wrapper`, pointing at this API's URL. The new version replaces the wrapper tool with one that uses it; the agent endpoint routes to the latest version.

In [4]:
connection = call('GET', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, 'news-mcp-apim', CONNECTION_API), ARM).json()
print(f"Connection news-mcp-apim: authType={connection['properties']['authType']}  target={connection['properties']['target']}")
assert connection['properties']['target'] == APIM_MCP_URL

agent = project.agents.create_version(agent_name=ident.AGENT_NAME, definition=PromptAgentDefinition(
    model=CHAT_MODEL, instructions=ident.AGENT_INSTRUCTIONS,
    tools=[ident.mcp_tool('news_mcp_apim', APIM_MCP_URL, connection['id'])]))
print(f'{agent.name} v{agent.version}: tool news_mcp_apim -> {APIM_MCP_URL}')

Connection news-mcp-apim: authType=AgentOnBehalfOfUser  target=https://apim-foundry-c2676f.azure-api.net/news-mcp/mcp
news-desk-agent v6: tool news_mcp_apim -> https://apim-foundry-c2676f.azure-api.net/news-mcp/mcp


## Sign in as user A and user B

Two device code sign-ins, as in 08-05c-01 and 08-05c-02. Each cell prints the account it expects and stops if the token belongs to someone else.

| Cell | Sign in as | Account | Method |
|---|---|---|---|
| first | user A | your admin account, the one `az login` uses | your usual password and multifactor method |
| second | user B | `news-sports-desk@{your tenant's default domain}`, in `.env` as `NEWS_TEST_USER_UPN` | Temporary Access Pass, in `.env` as `NEWS_TEST_USER_TAP` |

`grep -E 'NEWS_TEST_USER_(UPN|TAP)' .env` from the repository root shows user B's account and pass.

For user B:

1. Open a private browser window. A normal window offers the account you are already signed in with, and picking it signs in user A instead.
2. Open the address the cell prints and enter its code.
3. The page names `news-desk-client-{suffix}` and the location of the machine that runs the kernel. That is expected: continue.
4. If an account picker appears, choose **Use another account** and type user B's account.
5. If Entra asks for a password, choose **Other ways to sign in**, then **Temporary Access Pass**, and paste the pass.

The roles each user holds now come from Graph: user B lost `News.Read.Science` in 08-05c-02.

In [7]:
print('User A, your admin account:')
cred_a = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_a.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in; cred_a caches it
if checks.jwt_claims(token)['oid'] == TEST_USER_ID:
    raise RuntimeError('That sign-in was user B. Sign in as user A, your admin account, and re-run this cell.')

User A, your admin account:
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code F3P63T6HB to authenticate.


In [8]:
print(f"User B, {os.environ['NEWS_TEST_USER_UPN']}, in a private window, with the Temporary Access Pass in .env (NEWS_TEST_USER_TAP):")
cred_b = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_b.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in
if checks.jwt_claims(token)['oid'] != TEST_USER_ID:
    raise RuntimeError("That sign-in was not user B. A device code page reuses any account already signed in to the browser: open a private window, choose \"Use another account\", sign in as user B with the pass, and re-run this cell.")
print('Signed in as user B')

clients = {user: checks.agent_client(cred, PROJECT_ENDPOINT, ident.AGENT_NAME, BLUEPRINT_CLIENT_ID)
           for user, cred in (('A', cred_a), ('B', cred_b))}
ROLES = {'A': [authz.ALL_ROLE], 'B': roles_of(TEST_USER_ID)}
print(f"\nUser B holds {ROLES['B']}")

User B, {user}@{tenant}.onmicrosoft.com, in a private window, with the Temporary Access Pass in .env (NEWS_TEST_USER_TAP):
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code F2ZEKCLLA to authenticate.
Signed in as user B

User B holds ['News.Read.Sports', 'News.Read.Health']


## Step 4: The test matrix

The same prompts as on Track A, from [`news_mcp_checks.py`](news_mcp_checks.py). Both users see the same two tools, so user B asking for business is a call to `top_headlines(category=business)` that the policy answers with the denial text: `denied`, where Track A gave `hidden`. The text names user B's categories in GNews order, now without science.

The denial does not reach the model. Foundry treats a tool result with `isError: true` as a failed request: the agent endpoint answers HTTP 400 `tool_user_error`, "An error occurred invoking 'top_headlines': " followed by the policy's text. `checks.ask` turns that error into a `denied` outcome; an application calling the agent sees it as an error, with the text to show the user.

User B asking for science is `denied` too: `news-mcp-apim` has never fetched a token for user B, so the token Foundry obtains now is issued after the revocation.

In [9]:
print(f'News.Read.Science was revoked {minutes_since_revocation():.0f} minutes ago.\n')
cases = (*checks.base_cases('apim'), checks.science_case('denied'))
records = checks.run_matrix(clients, cases, ROLES)
print()
for user in ('A', 'B'):
    listed = next(r['listed'] for r in records if r['user'] == user and r['listed'])
    print(f'User {user} sees {listed}')

News.Read.Science was revoked 13 minutes ago.

A | Give me three business headlines    | listed  2 | called top_headlines            | articles    | expected articles | PASS
B | Give me three business headlines    | listed  0 | called top_headlines            | denied      | expected denied   | PASS
B | Give me three sports headlines      | listed  2 | called top_headlines            | articles    | expected articles | PASS
B | Give me three science headlines     | listed  0 | called top_headlines            | denied      | expected denied   | PASS
GNews calls so far: 2 (budget for a full run: 8)

User A sees ['search', 'top_headlines']
User B sees ['search', 'top_headlines']


**Observed on 2026-10-30.** User B's business and science requests both came back as HTTP 400 `tool_user_error` carrying the policy's text, so no reply formed and the response had no items: that is why those two lines list no tools. Both users' other requests listed `search` and `top_headlines`, and user A's business call returned articles, so the policy's read of the body (`preserveContent: true`) did not stop it reaching GNews.

Two things the gateway needed that the published material did not say. At `2025-09-01-preview` the service rejected `mcpProperties.endpoints` as an array, the shape the REST spec and its example give, and asked for a dictionary; this lab uses `2024-06-01-preview` with a backend, as the AI-Gateway samples do. And an MCP API without `backendId` answered `initialize` as "Azure API Management" with no tools. While that first version was live, Foundry served user A an empty tool list, and kept doing so for user A's token after the API was fixed, even on a new agent version; a new sign-in cleared it.

## Step 5: The denial without the agent

User B's own token for `api://news-mcp-{suffix}/News.Read` goes straight to the gateway with the call the agent made in Step 4. The policy answers it with `isError: true` and the denial text, and GNews never sees the request.

Only a denied call can be sent this way. GNews keeps a session per client: a `tools/list` or an allowed `tools/call` without the `mcp-session-id` from an `initialize` comes back as HTTP 400 "Missing session ID". Foundry runs that handshake on every response, through the gateway.

In [10]:
token_b = cred_b.get_token(f'{authz.app_id_uri(SUFFIX)}/{authz.SCOPE}').token
result = checks.call_tool_direct(APIM_MCP_URL, token_b, 'top_headlines', {'category': 'business', 'max': 3})
denial = result['content'][0]['text']
print(f"User B calls top_headlines(category=business) at the gateway: isError={result['isError']}\n  {denial}")
assert result['isError'] and denial == authz.denial_message(ROLES['B'])

User B calls top_headlines(category=business) at the gateway: isError=True
  Your access covers: sports, health. Do not retry other categories.


## Step 6: Has the revocation reached Track A?

The agent endpoint runs one particular version when the request names it in `agent_reference`. The cell finds the newest version with the wrapper tool and asks it for science as user B. The answer depends on the token Foundry presents to the wrapper:

| Outcome | What it means |
|---|---|
| `articles` | Foundry reused a token for user B obtained before the revocation; it still carries `News.Read.Science` until it expires |
| `hidden` | Foundry obtained a new token; the wrapper listed only sports and health |

Either is a valid result; the case is not required. The wrapper's log line for the request's `tools/list` shows the roles in the token it received.

In [11]:
def wrapper_events(kind=None, tail=300):
    """The Track A wrapper's recent JSON events from the container app's console log."""
    r = subprocess.run(['az', 'containerapp', 'logs', 'show', '-n', TRACK_A_APP, '-g', TRACK_A_RG, '--type', 'console',
                        '--tail', str(tail), '--format', 'json'], capture_output=True, text=True)
    return checks.wrapper_events(r.stdout.splitlines(), kind)


versions = [v.as_dict() for v in project.agents.list_versions(agent_name=ident.AGENT_NAME)]
track_a = ident.latest_version_with_tool(versions, 'news_mcp_wrapper')
print(f"Track A: {ident.AGENT_NAME} v{track_a}; News.Read.Science revoked {minutes_since_revocation():.0f} minutes ago\n")
checks.run_matrix(clients, [checks.science_case('hidden', required=False)], ROLES,
                  extra_body=ident.agent_reference(ident.AGENT_NAME, track_a))
time.sleep(10)  # the log stream lags a few seconds
seen = [e for e in wrapper_events('tools/list') if e['oid'] == TEST_USER_ID][-1]
print(f"\nThe wrapper's last tools/list for user B: roles {seen['roles']}, tools {seen['tools']}")

Track A: news-desk-agent v4; News.Read.Science revoked 13 minutes ago

B | Give me three science headlines     | listed  0 | called -                        | hidden      | expected hidden   | PASS (not required)
GNews calls so far: 2 (budget for a full run: 8)

The wrapper's last tools/list for user B: roles ['News.Read.Sports', 'News.Read.Health'], tools ['get_sports_headlines', 'get_health_headlines']


**Observed on 2026-10-30.** 08-05c-02 revoked `News.Read.Science` at 21:01:51 UTC, 3 seconds after Foundry had called the wrapper with a token for user B. Thirteen minutes later the Track A version no longer offered science to user B: the wrapper's `tools/list` saw a token with `News.Read.Sports` and `News.Read.Health` only. This notebook's user B signed in after the revocation; a request with 08-05c-02's user token, issued before it, was not served science either. Foundry did not keep presenting a token from before the revocation for these 13 minutes.

## Cleanup

The full teardown is the Cleanup section of [08-05c-01](08-05c-01-identity-and-agent.ipynb), which removes this notebook's API, backend, named value and connection too. To remove only Track B, uncomment the cell: the API (its policy goes with it), the backend, the named value, and the `news-mcp-apim` connection. The `news-desk-agent` version that uses the connection stays until 08-05c-01 deletes the agent.

In [ ]:
# for path in ('apis/news-mcp', 'backends/gnews-mcp', 'namedValues/gnews-api-key'):
#     print(path, call('DELETE', f'{APIM}/{path}?api-version={APIM_API_VERSION}', ARM).status_code)
# print('news-mcp-apim', call('DELETE', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, 'news-mcp-apim', CONNECTION_API), ARM).status_code)